# Day 038 Solution — Exploratory Data Analysis

Distribution profiling, top-group ranking, correlation, pivot tables, and a full EDA report dict. All data defined inline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


import pandas as pd

def top_groups(df: pd.DataFrame, group_col: str, value_col: str,
               n: int = 5) -> pd.DataFrame:
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


import pandas as pd

def correlation_summary(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    corr   = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)


import pandas as pd

def pivot_summary(df: pd.DataFrame, index: str, columns: str,
                  values: str, aggfunc: str = 'mean') -> pd.DataFrame:
    piv = pd.pivot_table(
        df, values=values, index=index, columns=columns,
        aggfunc=aggfunc, fill_value=0,
    )
    piv.columns.name = None
    return piv.reset_index()


import pandas as pd

def eda_report(df: pd.DataFrame) -> dict:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape':           df.shape,
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'category_counts': {col: df[col].value_counts().to_dict() for col in cat_cols},
        'correlations':    df[num_cols].corr().round(4).to_dict() if len(num_cols) > 1 else {},
    }

## Step 1 — Load & Inspect

In [ ]:
RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
df = pd.read_csv(io.StringIO(RETAIL_CSV))
df['revenue'] = df['price'] * df['quantity']
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(df.head())

assert df.shape == (12, 7)
assert 'revenue' in df.columns

## Step 2 — Distribution Summary

In [ ]:
# Numeric column
rev_dist = distribution_summary(df, 'revenue')
print('Revenue distribution:')
for k, v in rev_dist.items():
    print(f'  {k:12s}: {v}')

assert rev_dist['count'] == 12
assert rev_dist['null_count'] == 0
assert rev_dist['q25'] <= rev_dist['median'] <= rev_dist['q75']

# Categorical column
prod_dist = distribution_summary(df, 'product')
print(f'\nProduct distribution: unique={prod_dist["unique"]}, top={prod_dist["top"]!r}')

assert prod_dist['unique'] == 4
assert prod_dist['top'] == 'Widget'  # Widget appears 4 times (most frequent)

## Step 3 — Top Groups

In [ ]:
top3 = top_groups(df, 'product', 'revenue', n=3)
print('Top 3 products by revenue:')
print(top3.to_string(index=False))

assert len(top3) == 3
assert top3.iloc[0]['product'] == 'Gadget'  # Gadget total=1800 (top by revenue)
assert top3.iloc[0]['total'] == 1800.0

# Top regions
top_regions = top_groups(df, 'region', 'revenue', n=4)
print('\nRevenue by region:')
print(top_regions.to_string(index=False))

## Step 4 — Correlation

In [ ]:
corr = correlation_summary(df, 'revenue')
print('Correlations with revenue:')
print(corr.to_string(index=False))

assert list(corr.columns) == ['feature', 'correlation']
assert 'revenue' not in corr['feature'].values
abs_vals = corr['correlation'].abs().tolist()
assert abs_vals == sorted(abs_vals, reverse=True)

## Step 5 — Pivot Table

In [ ]:
piv = pivot_summary(df, 'product', 'region', 'revenue', 'sum')
print('Revenue by product × region:')
print(piv.to_string(index=False))

assert 'product' in piv.columns
assert piv.isnull().sum().sum() == 0

# Cross-tabulation shortcut (counts)
print('\nOrder count by product × region:')
print(pd.crosstab(df['product'], df['region']))

## Step 6 — Full EDA Report

In [ ]:
report = eda_report(df)
print(f'Shape           : {report["shape"]}')
print(f'Null counts     : {report["null_counts"]}')
print(f'Numeric cols    : {list(report["numeric_summary"].keys())}')
print(f'Category cols   : {list(report["category_counts"].keys())}')
print(f'Corr matrix cols: {list(report["correlations"].keys())}')

assert report['shape'] == df.shape
assert 'revenue' in report['numeric_summary']
assert 'product' in report['category_counts']

print('\nExploratory Data Analysis complete!')